In [ ]:
!pip uninstall -y -q transformers
!pip install -q transformers==4.51.3 accelerate qwen-vl-utils

In [ ]:
import transformers, torch
from pathlib import Path
print("transformers:", transformers.__version__)     # لازم 4.51.3
print("cuda:", torch.cuda.is_available())            # لازم True
print("صور:", len(list(Path("pages").glob("*.png")))) # لازم 49

transformers: 5.13.1
cuda: True
صور: 0


In [ ]:
import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoProcessor
from qwen_vl_utils import process_vision_info

model_path = snapshot_download("dots-studio/dots.ocr", local_dir="./weights/DotsOCR")

cap = torch.cuda.get_device_capability()[0]
dtype = torch.bfloat16 if cap >= 8 else torch.float16
attn = "flash_attention_2" if cap >= 8 else "sdpa"
print(f"dtype={dtype}  attn={attn}")

model = AutoModelForCausalLM.from_pretrained(
    model_path, attn_implementation="sdpa", torch_dtype=torch.bfloat16,
    device_map="auto", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model.eval()
print("جاهز")

Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

dtype=torch.float16  attn=sdpa
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementation 
flash attention not available! fallback to eager implementa

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

جاهز


In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
print(f"فاضي: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

فاضي: 2.3 GB


In [ ]:
import json, re, torch
from pathlib import Path
from qwen_vl_utils import process_vision_info

RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
MAX_PIXELS = 1000000

LAYOUT = """Please output the layout information from the PDF image, including each layout element's bbox, its category, and the corresponding text content within the bbox.

1. Bbox format: [x1, y1, x2, y2]

2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title'].

3. Text Extraction & Formatting Rules:
    - Picture: For the 'Picture' category, the text field should be omitted.
    - Formula: Format its text as LaTeX.
    - Table: Format its text as HTML.
    - All Others (Text, Title, etc.): Format their text as Markdown.

4. Constraints:
    - The output text must be the original text from the image, with no translation.
    - All layout elements must be sorted according to human reading order.

5. Final Output: The entire output must be a single JSON object.
"""

OCR_ONLY = "Extract the text content from this image."

def looks_stuck(t):
    if not t: return True
    if re.search(r"(.{6,60}?)\1{4,}", t): return True
    return len(set(t[-400:])) < 8 and len(t) > 2000

def run_page(img, prompt, max_tokens=6000):
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": str(img), "max_pixels": MAX_PIXELS},
        {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ii, vi = process_vision_info(msgs)
    inputs = processor(text=[text], images=ii, videos=vi,
                       padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

pages = sorted(Path("pages").glob("*.png"))
for p in pages:
    dst = RESULTS / f"{p.stem}.txt"
    if dst.exists(): continue          # يكمّل لو الجلسة قطعت
    r = run_page(p, LAYOUT)
    if looks_stuck(r):
        print(f"  ↻ {p.name}: تكرار — إعادة")
        r = run_page(p, OCR_ONLY, 3000)
    dst.write_text(r, encoding="utf-8")
    print(f"✓ {p.name}  ({len(r)} حرف)")
print("خلص!")

خلص!


In [ ]:
print(run_page(Path("/content/page_018.png"), LAYOUT)[:1500])

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:965: FutureWarning: Both `num_logits_to_keep` and `logits_to_keep` are set for `DotsOCRForCausalLM.forward`. Using `logits_to_keep=1` and ignoring deprecated `num_logits_to_keep=None`.
  output = func(self, *args, **kwargs)


[{"bbox": [273, 294, 799, 530], "category": "Text", "text": "يترك الذكور السرب عادةً حين تتراوح أعمارهم بين 4 و 21 سنة. وعندها ينضمُون إلى مجموعات من lives of the men of the 21st century، ويصل عددُهم فيها 50. الحيتان الذكور فقط، وقد ذكرنا أنَّ هناكُم ذكرًا. وفي المعتاد تسبُح الذكور الأكبر سنًا والأضخم حجمًا وحدها."}, {"bbox": [88, 1052, 118, 1076], "category": "Page-footer", "text": "17"}]


In [ ]:
import json

def to_text(raw, skip={"Picture", "Page-header", "Page-footer"}):
    try:
        blocks = json.loads(raw[raw.index("["):raw.rindex("]") + 1])
    except Exception:
        return raw.strip()
    out = []
    for b in blocks:
        if b.get("category") in skip:
            continue
        t = (b.get("text") or "").strip()
        if t:
            out.append(t)
    return "\n\n".join(out)

raw = run_page(Path("/content/page_018.png"), LAYOUT)
print(to_text(raw))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


يترك الذكور السرب عادةً حين تتراوح أعمارهم بين 4 و 21 سنة. وعندها ينضمُون إلى مجموعات من lives of the men of the 21st century، ويصل عددُهم فيها 50. الحيتان الذكور فقط، وقد ذكرنا أنَّ هناكُم ذكرًا. وفي المعتاد تسبُح الذكور الأكبر سنًا والأضخم حجمًا وحدها.


In [18]:
import json, re
from pathlib import Path

RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
SKIP = {"Picture", "Page-header", "Page-footer"}

def parse(raw):
    """يحوّل مخرجات الموديل لليستة بلوكات. لو مش JSON، يرجّعه كبلوك نص واحد."""
    try:
        return json.loads(raw[raw.index("["):raw.rindex("]") + 1])
    except Exception:
        return [{"bbox": None, "category": "Text", "text": raw.strip()}]

book = []
for p in sorted(Path("pages").glob("*.png")):
    cache = RESULTS / f"{p.stem}.json"

    if cache.exists():
        page = json.loads(cache.read_text(encoding="utf-8"))
    else:
        raw = run_page(p, LAYOUT)
        if looks_stuck(raw):
            raw = run_page(p, OCR_ONLY, 3000)
        blocks = parse(raw)
        page = {
            "page": int(p.stem.split("_")[1]),
            "image": p.name,
            "blocks": blocks,
            "text": "\n\n".join(
                b["text"].strip() for b in blocks
                if b.get("category") not in SKIP and (b.get("text") or "").strip()
            ),
        }
        cache.write_text(json.dumps(page, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"✓ {p.name}  ({len(page['text'])} حرف)")

    book.append(page)

Path("book.json").write_text(
    json.dumps(book, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"\nخلص — {len(book)} صفحة في book.json")


خلص — 0 صفحة في book.json


In [28]:
import json, re, torch
from pathlib import Path
from qwen_vl_utils import process_vision_info

PAGES_DIR = Path("/content/page")
MAX_PIXELS = 1000000
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
SKIP = {"Picture", "Page-header", "Page-footer"}

LAYOUT = """Please output the layout information from the PDF image, including each layout element's bbox, its category, and the corresponding text content within the bbox.

1. Bbox format: [x1, y1, x2, y2]

2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title'].

3. Text Extraction & Formatting Rules:
    - Picture: For the 'Picture' category, the text field should be omitted.
    - Formula: Format its text as LaTeX.
    - Table: Format its text as HTML.
    - All Others (Text, Title, etc.): Format their text as Markdown.

4. Constraints:
    - The output text must be the original text from the image, with no translation.
    - All layout elements must be sorted according to human reading order.

5. Final Output: The entire output must be a single JSON object.
"""

OCR_ONLY = "Extract the text content from this image."


def looks_stuck(t):
    if not t: return True
    if re.search(r"(.{6,60}?)\1{4,}", t): return True
    return len(set(t[-400:])) < 8 and len(t) > 2000


def run_page(img, prompt, max_tokens=6000):
    msgs = [{"role": "user", "content": [
        {"type": "image", "image": str(img), "max_pixels": MAX_PIXELS},
        {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ii, vi = process_vision_info(msgs)
    inputs = processor(text=[text], images=ii, videos=vi,
                       padding=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
    return processor.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def parse(raw):
    try:
        return json.loads(raw[raw.index("["):raw.rindex("]") + 1])
    except Exception:
        return [{"bbox": None, "category": "Text", "text": raw.strip()}]


pages = sorted(PAGES_DIR.glob("page_*.png"))
assert len(pages) == 49, f"لقيت {len(pages)} صورة بس — المفروض 49. شوف: !ls {PAGES_DIR}"
print(f"لقيت {len(pages)} صورة\n")

book = []
for i, p in enumerate(pages, 1):
    cache = RESULTS / f"{p.stem}.json"

    if cache.exists():
        page = json.loads(cache.read_text(encoding="utf-8"))
    else:
        raw = run_page(p, LAYOUT)
        if looks_stuck(raw):
            print("  ↻ تكرار — إعادة")
            raw = run_page(p, OCR_ONLY, 3000)
        blocks = parse(raw)
        page = {
            "page": int(p.stem.split("_")[1]),
            "image": p.name,
            "blocks": blocks,
            "text": "\n\n".join(
                b["text"].strip() for b in blocks
                if b.get("category") not in SKIP and (b.get("text") or "").strip()
            ),
        }
        cache.write_text(json.dumps(page, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"[{i}/49] {p.name}  {len(page['text'])} حرف")

    book.append(page)

Path("book.json").write_text(json.dumps(book, ensure_ascii=False, indent=2), encoding="utf-8")
Path("book.txt").write_text(
    "\n\n---\n\n".join(f"[صفحة {pg['page']}]\n\n{pg['text']}" for pg in book),
    encoding="utf-8",
)
print(f"\nخلص — {len(book)} صفحة في book.json و book.txt")

لقيت 49 صورة



Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[1/49] page_001.png  62 حرف


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[2/49] page_002.png  39 حرف
[3/49] page_003.png  1286 حرف


...

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[43/49] page_043.png  598 حرف
[44/49] page_044.png  93 حرف


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[45/49] page_045.png  133 حرف


OSError: image file is truncated

In [30]:
!zip -qr results_backup.zip results/
from google.colab import files
files.download("results_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
import json
from pathlib import Path

files = sorted(Path("results").glob("page_*.json"))
print(f"{len(files)} صفحة في الكاش\n")

pages = [json.loads(f.read_text(encoding="utf-8")) for f in files]

# ملخص سريع: كل صفحة وكام حرف فيها
for pg in pages:
    flag = "  ⚠️ فاضية" if len(pg["text"]) < 20 else ""
    print(f"صفحة {pg['page']:>2}: {len(pg['text']):>4} حرف{flag}")

45 صفحة في الكاش

صفحة  1:   62 حرف
صفحة  2:   39 حرف
صفحة  3: 1286 حرف
صفحة  4:  255 حرف
صفحة  5:  266 حرف
صفحة  6:  191 حرف
صفحة  7:  118 حرف
صفحة  8:   12 حرف  ⚠️ فاضية
صفحة  9:   13 حرف  ⚠️ فاضية
صفحة 10:  358 حرف
صفحة 11:   63 حرف
صفحة 12:  165 حرف
صفحة 13:  154 حرف
صفحة 14:  317 حرف
صفحة 15:  438 حرف
صفحة 16:  397 حرف
صفحة 17:  183 حرف
صفحة 18:  254 حرف
صفحة 19:   27 حرف
صفحة 20:  349 حرف
صفحة 21:  459 حرف
صفحة 22:  126 حرف
صفحة 23:  324 حرف
صفحة 24:  256 حرف
صفحة 25:  299 حرف
صفحة 26:    0 حرف  ⚠️ فاضية
صفحة 27:  178 حرف
صفحة 28:  289 حرف
صفحة 29:   17 حرف  ⚠️ فاضية
صفحة 30:  148 حرف
صفحة 31:  158 حرف
صفحة 32:  252 حرف
صفحة 33:  467 حرف
صفحة 34:  483 حرف
صفحة 35:  380 حرف
صفحة 36:  241 حرف
صفحة 37:  390 حرف
صفحة 38:  374 حرف
صفحة 39:  115 حرف
صفحة 40:  726 حرف
صفحة 41:  264 حرف
صفحة 42:  300 حرف
صفحة 43:  598 حرف
صفحة 44:   93 حرف
صفحة 45:  133 حرف


In [32]:
Path("book.json").write_text(json.dumps(pages, ensure_ascii=False, indent=2), encoding="utf-8")
Path("book.txt").write_text(
    "\n\n---\n\n".join(f"[صفحة {pg['page']}]\n\n{pg['text']}" for pg in pages),
    encoding="utf-8",
)

from google.colab import files
files.download("book.json")
files.download("book.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>